In [1]:
import numpy as np
bohr_to_ang = 0.529177249 # Angstrom / Bohr

# Equilibrium Atomic Positions

From https://next-gen.materialsproject.org/materials/mp-1191749 we have the 

- Conventional unit cell 
    - $a = b = c = 10.19$ Angstroms, 
    - $\alpha = \beta = \gamma = 90$ degrees. 

- Primitive unit cell
    - $a = b = c = 7.20$ Angstroms,
    - $\alpha = \beta = \gamma = 60$ degrees.

In [2]:
a0 = 10.19 # Angstrom, conventional cubic lattice constant

a1 = np.array([0, a0/2, a0/2])
a2 = np.array([a0/2, 0, a0/2])
a3 = np.array([a0/2, a0/2, 0])

print("CELL_PARAMETERS angstrom")
print(f"{a1[0]:.6f} {a1[1]:.6f} {a1[2]:.6f}")
print(f"{a2[0]:.6f} {a2[1]:.6f} {a2[2]:.6f}")
print(f"{a3[0]:.6f} {a3[1]:.6f} {a3[2]:.6f}")

print("\nLattice vector length:")

print("|a0| = {:.6f} Angstrom".format(np.linalg.norm(a1)) )
print("|a1| = {:.6f} Angstrom".format(np.linalg.norm(a2)) )
print("|a2| = {:.6f} Angstrom".format(np.linalg.norm(a3)) )

CELL_PARAMETERS angstrom
0.000000 5.095000 5.095000
5.095000 0.000000 5.095000
5.095000 5.095000 0.000000

Lattice vector length:
|a0| = 7.205418 Angstrom
|a1| = 7.205418 Angstrom
|a2| = 7.205418 Angstrom


## Atomic positions in the primitive cell

From SMODES output we have (in units of Angstroms):

```text
Vectors defining superlattice
   0.00000   5.09500   5.09500
   5.09500   0.00000   5.09500
   5.09500   5.09500   0.00000
atom  type   position
  1   Y      5.09500   5.09500   5.09500
  2   Y      5.09500   2.54750   2.54750
  3   Y      2.54750   5.09500   2.54750
  4   Y      2.54750   2.54750   5.09500
  5   Ir     0.00000   0.00000   0.00000
  6   Ir     0.00000   2.54750   2.54750
  7   Ir     2.54750   0.00000   2.54750
  8   Ir     2.54750   2.54750   0.00000
  9   O      3.82125   3.82125   3.82125
 10   O      6.36875   6.36875   6.36875
 11   O      0.00000   1.27375   1.27375
 12   O      2.54750   1.27375   1.27375
 13   O      1.27375   1.27375   0.00000
 14   O      1.27375   1.27375   2.54750
 15   O      1.27375   0.00000   1.27375
 16   O      1.27375   2.54750   1.27375
 17   O      3.82125   0.00000   3.82125
 18   O      3.82125   7.64250   3.82125
 19   O      0.00000   3.82125   3.82125
 20   O      7.64250   3.82125   3.82125
 21   O      3.82125   3.82125   0.00000
 22   O      3.82125   3.82125   7.64250
```

In [3]:
import numpy as np

# ============================================================
# Y2Ir2O7 in Fd-3m (#227)
# Non-pymatgen, hardcoded orbit generation for the occupied
# Wyckoff sites in the MP convention:
#
#   Y  : 16d  (1/8, 7/8, 3/8)
#   Ir : 16c  (5/8, 7/8, 3/8)
#   O' : 8b   (1/4, 3/4, 1/4)
#   O  : 48f  (0, 0, z)
#
# Then convert to the SMODES primitive cell
# ============================================================

a = 10.19
z = 0.212932

# ----------------------------
# Helpers
# ----------------------------
def wrap01(x, tol=1e-10):
    x = np.mod(np.array(x, dtype=float), 1.0)
    x[np.abs(x) < tol] = 0.0
    x[np.abs(x - 1.0) < tol] = 0.0
    return x

def frac_equal(f1, f2, tol=1e-8):
    d = wrap01(f1 - f2)
    d = np.minimum(d, 1.0 - d)
    return np.linalg.norm(d) < tol

def unique_fracs(fracs, tol=1e-8):
    out = []
    for f in fracs:
        f = wrap01(f)
        if not any(frac_equal(f, g, tol) for g in out):
            out.append(f)
    return out

def frac_to_cart_conv(frac, a):
    """Conventional cubic fractional -> Cartesian."""
    frac = np.asarray(frac, dtype=float)
    return a * frac

def prim_frac_to_cart(frac, A_prim):
    """Primitive fractional -> Cartesian, lattice vectors as rows."""
    return np.asarray(frac) @ A_prim

def cart_to_prim_frac(cart, A_prim):
    """Cartesian -> primitive fractional."""
    return np.asarray(cart) @ np.linalg.inv(A_prim)

def choose_in_primitive_cell(frac_list, tol=1e-8):
    """
    Given many equivalent primitive fractional coordinates, choose one in [0,1).
    """
    out = []
    for f in frac_list:
        f = wrap01(f)
        if not any(frac_equal(f, g, tol) for g in out):
            out.append(f)
    return out

# ----------------------------
# Conventional cubic lattice
# ----------------------------
A_conv = np.array([
    [a, 0.0, 0.0],
    [0.0, a, 0.0],
    [0.0, 0.0, a],
])

# SMODES primitive lattice
A_prim = np.array([
    [0.0, a/2, a/2],
    [a/2, 0.0, a/2],
    [a/2, a/2, 0.0],
])

# F-centering translations
F_trans = [
    np.array([0.0, 0.0, 0.0]),
    np.array([0.0, 0.5, 0.5]),
    np.array([0.5, 0.0, 0.5]),
    np.array([0.5, 0.5, 0.0]),
]

# ============================================================
# Orbit generators in the MP conventional setting
# ============================================================

def orbit_16c_MP():
    """
    MP representative: (5/8, 7/8, 3/8)
    In this setting, the full 16c orbit generated from that rep is:
      base tetrahedron + F-centering
    """
    base = [
        np.array([5/8, 7/8, 3/8]),
        np.array([5/8, 5/8, 1/8]),
        np.array([3/8, 7/8, 1/8]),
        np.array([3/8, 5/8, 3/8]),
    ]
    pts = []
    for t in F_trans:
        for b in base:
            pts.append(wrap01(b + t))
    return unique_fracs(pts)

def orbit_16d_MP():
    """
    MP representative: (1/8, 7/8, 3/8)
    """
    base = [
        np.array([1/8, 7/8, 3/8]),
        np.array([1/8, 5/8, 1/8]),
        np.array([3/8, 7/8, 1/8]),
        np.array([3/8, 5/8, 3/8]),
    ]
    pts = []
    for t in F_trans:
        for b in base:
            pts.append(wrap01(b + t))
    return unique_fracs(pts)

def orbit_8b_MP():
    """
    MP representative: (1/4, 3/4, 1/4)
    """
    base = [
        np.array([1/4, 3/4, 1/4]),
        np.array([3/4, 1/4, 1/4]),
    ]
    pts = []
    for t in F_trans:
        for b in base:
            pts.append(wrap01(b + t))
    return unique_fracs(pts)

def orbit_48f_MP(z):
    """
    48f in the MP setting, using representative (0,0,z).
    For this specific setting, the orbit can be generated from permutations/signs:
        (0,0, z), (0, z, 0), (z, 0, 0),
        (0,0,-z), ...
    plus offsets by (1/4,1/4,1/4)-type combinations appropriate to Fd-3m.

    A compact way for this specific case:
      generate the 12-site primitive-cell oxygen set in the SMODES-compatible
      primitive cell, then expand by F-centering back to 48 conventional sites.
    """
    # First construct the standard 12-site primitive 48f set
    # in the SMODES primitive cell using the known parameter relation.
    q = z - 1/8
    p = 1/4 - q

    prim12 = [
        [p, q, q],
        [q, p, p],
        [q, q, p],
        [p, p, q],
        [q, p, q],
        [p, q, p],
        [1-q, 1-p, 1-q],
        [1-p, 1-q, 1-p],
        [1-p, 1-q, 1-q],
        [1-q, 1-p, 1-p],
        [1-q, 1-q, 1-p],
        [1-p, 1-p, 1-q],
    ]

    # Convert those primitive fractional coords to Cartesian,
    # then to conventional fractional coords.
    conv_pts = []
    for fp in prim12:
        cart = prim_frac_to_cart(fp, A_prim)
        fc = cart / a
        conv_pts.append(wrap01(fc))

    # Add F-centering images and deduplicate
    pts = []
    for t in F_trans:
        for c in conv_pts:
            pts.append(wrap01(c + t))
    return unique_fracs(pts)

# ============================================================
# Generate full conventional structure from MP reps
# ============================================================

Y16d = orbit_16d_MP()
Ir16c = orbit_16c_MP()
O8b  = orbit_8b_MP()
O48f = orbit_48f_MP(z)

print("Conventional-cell orbit sizes:")
print("Y  16d:", len(Y16d))
print("Ir 16c:", len(Ir16c))
print("O  8b :", len(O8b))
print("O 48f:", len(O48f))

# ============================================================
# Convert full conventional structure to primitive cell
# ============================================================

def conventional_orbit_to_primitive(orbits, species_labels):
    """
    Convert all conventional fractional positions to primitive fractional positions,
    then choose unique representatives in the primitive cell.
    """
    out = []
    for sp, orb in zip(species_labels, orbits):
        prim_list = []
        for fc in orb:
            cart = frac_to_cart_conv(fc, a)
            fp = cart_to_prim_frac(cart, A_prim)
            prim_list.append(wrap01(fp))
        prim_unique = choose_in_primitive_cell(prim_list)
        for fp in prim_unique:
            out.append((sp, fp))
    return out

prim_sites = conventional_orbit_to_primitive(
    [Y16d, Ir16c, O8b, O48f],
    ["Y", "Ir", "O", "O"]
)

# split by species for nicer display
Ys  = [fp for sp, fp in prim_sites if sp == "Y"]
Irs = [fp for sp, fp in prim_sites if sp == "Ir"]
Os  = [fp for sp, fp in prim_sites if sp == "O"]

print("\nPrimitive-cell counts:")
print("Y :", len(Ys))
print("Ir:", len(Irs))
print("O :", len(Os))

# ============================================================
# Print primitive coordinates in the SMODES cell
# ============================================================

def sort_rows(rows):
    return sorted([wrap01(r) for r in rows], key=lambda x: tuple(np.round(x, 8)))

print("\nPrimitive fractional coordinates:")
for sp, rows in [("Y", Ys), ("Ir", Irs), ("O", Os)]:
    for r in sort_rows(rows):
        print(f"{sp:2s}  {r[0]:10.6f} {r[1]:10.6f} {r[2]:10.6f}")

print("\nPrimitive Cartesian coordinates (angstrom):")
for sp, rows in [("Y", Ys), ("Ir", Irs), ("O", Os)]:
    for r in sort_rows(rows):
        c = prim_frac_to_cart(r, A_prim)
        print(f"{sp:2s}  {c[0]:10.5f} {c[1]:10.5f} {c[2]:10.5f}")

# ============================================================
# Show the SMODES reference oxygens and the A1g shift
# ============================================================

delta = a * (z - 1/8)

R_ref = {
    11: np.array([0.00000, 1.27375, 1.27375]),
    12: np.array([2.54750, 1.27375, 1.27375]),
    13: np.array([1.27375, 1.27375, 0.00000]),
    14: np.array([1.27375, 1.27375, 2.54750]),
    15: np.array([1.27375, 0.00000, 1.27375]),
    16: np.array([1.27375, 2.54750, 1.27375]),
    17: np.array([3.82125, 0.00000, 3.82125]),
    18: np.array([3.82125, 7.64250, 3.82125]),
    19: np.array([0.00000, 3.82125, 3.82125]),
    20: np.array([7.64250, 3.82125, 3.82125]),
    21: np.array([3.82125, 3.82125, 0.00000]),
    22: np.array([3.82125, 3.82125, 7.64250]),
}

d_A1g = {
    11: np.array([+1, 0, 0]),
    12: np.array([-1, 0, 0]),
    13: np.array([0, 0, +1]),
    14: np.array([0, 0, -1]),
    15: np.array([0, +1, 0]),
    16: np.array([0, -1, 0]),
    17: np.array([0, -1, 0]),
    18: np.array([0, +1, 0]),
    19: np.array([-1, 0, 0]),
    20: np.array([+1, 0, 0]),
    21: np.array([0, 0, -1]),
    22: np.array([0, 0, +1]),
}

print("\nSMODES oxygen reference -> equilibrium mapping:")
print(f"delta = a * (z - 1/8) = {delta:.5f} Å\n")
for i in range(11, 23):
    req = R_ref[i] + delta * d_A1g[i]
    print(f"{i:2d}: ref {R_ref[i]}  +  delta*{d_A1g[i]}  =  eq {np.round(req, 5)}")

Conventional-cell orbit sizes:
Y  16d: 16
Ir 16c: 16
O  8b : 4
O 48f: 48

Primitive-cell counts:
Y : 4
Ir: 4
O : 13

Primitive fractional coordinates:
Y     0.125000   0.625000   0.625000
Y     0.625000   0.125000   0.625000
Y     0.625000   0.625000   0.125000
Y     0.625000   0.625000   0.625000
Ir    0.125000   0.125000   0.125000
Ir    0.625000   0.125000   0.125000
Ir    0.625000   0.125000   0.625000
Ir    0.625000   0.625000   0.125000
O     0.087932   0.087932   0.162068
O     0.087932   0.162068   0.087932
O     0.087932   0.162068   0.162068
O     0.162068   0.087932   0.087932
O     0.162068   0.087932   0.162068
O     0.162068   0.162068   0.087932
O     0.750000   0.750000   0.750000
O     0.837932   0.837932   0.912068
O     0.837932   0.912068   0.837932
O     0.837932   0.912068   0.912068
O     0.912068   0.837932   0.837932
O     0.912068   0.837932   0.912068
O     0.912068   0.912068   0.837932

Primitive Cartesian coordinates (angstrom):
Y      6.36875    3.82125  



The starting point is the Materials Project / Bilbao conventional description of pyrochlore $Y_2 Ir_2 O_7$ in space group $Fd\bar{3}m$ (#227), with conventional cubic lattice constant $a=10.19$ Å and Wyckoff representatives

- Y on $16d$
- Ir on $16c$
- O' on $8b$
- O on $48f$ with oxygen parameter $z=0.212932$

From these Wyckoff representatives, the full conventional structure is generated by applying the space-group symmetry operations. That conventional structure is then rewritten in the primitive fcc cell with lattice vectors

$$
\mathbf a_1=(0,a/2,a/2),\qquad
\mathbf a_2=(a/2,0,a/2),\qquad
\mathbf a_3=(a/2,a/2,0).
$$

This primitive-cell representation is not unique: different valid origin choices and different symmetry-equivalent representatives of the same Wyckoff orbit can lead to coordinate lists that look different even though they describe the same crystal.

### Why the oxygen coordinates are subtle

The main subtlety is the $48f$ oxygen site. In the SMODES output, the oxygen atoms labeled 11–22 are printed in a high-symmetry reference configuration. To recover the **physical equilibrium structure**, these oxygen positions must be shifted along the $A_{1g}$ breathing-mode directions so that the correct oxygen internal parameter is restored.

The required shift amplitude is

$$
\Delta = a\left(z-\frac18\right)
=10.19\,(0.212932-0.125)=0.89603\ \text{Å}.
$$

Thus the equilibrium structure used for the frozen-phonon calculation is obtained by keeping atoms 1–10 as printed by SMODES and correcting atoms 11–22 by this $A_{1g}$ displacement. This gives a structure in the **same primitive cell and atom numbering convention as SMODES**, which is exactly what is needed for a one-to-one mapping between SMODES mode vectors and the atoms displaced in the DFT calculation.

### Why this structure is physically correct

Several independent checks support that this is the correct equilibrium structure:

1. **Recovered symmetry**  
   When the SMODES-indexed equilibrium structure is standardized, it returns to the same cubic space group,
   $$
   Fd\bar{3}m \quad (\#227),
   $$
   with the same lattice parameter $a=10.19$ Å.

2. **Correct pyrochlore oxygen parameter**  
   The standardized conventional structure contains oxygen coordinates that encode the same internal parameter $z=0.212932$, although not necessarily in the same representative form as the Materials Project page. This is expected, because the same Wyckoff orbit can be written in multiple symmetry-equivalent ways depending on the chosen origin and representative.

3. **Correct local IrO$_6$ geometry**  
   For the Ir atom at the origin, the six nearest oxygen neighbors are
   atoms 11, 13, 15, 17, 19, and 21, all at the same distance:
   \[
   d_{\mathrm{Ir-O}} = 2.011900\ \text{Å}.
   \]
   This is exactly the behavior expected for the equilibrium pyrochlore structure: a regular first oxygen coordination shell around Ir, with the oxygen internal parameter fixed by the Materials Project / Bilbao value.

These checks are stronger than literal coordinate matching to the Materials Project page, because the Materials Project gives the structure in a conventional crystallographic setting, whereas the frozen-phonon calculation is performed in a primitive cell with the SMODES atom ordering.


In [4]:
import numpy as np

a = 10.19
delta = a * (0.212932 - 0.125)   # 0.89603 Å

labels = list(range(1, 23))
species = {
    1:"Y",2:"Y",3:"Y",4:"Y",
    5:"Ir1",6:"Ir2",7:"Ir3",8:"Ir4",
    9:"O",10:"O",11:"O",12:"O",13:"O",14:"O",15:"O",16:"O",17:"O",18:"O",19:"O",20:"O",21:"O",22:"O"
}

R_ref = {
 1: np.array([5.09500,5.09500,5.09500]),
 2: np.array([5.09500,2.54750,2.54750]),
 3: np.array([2.54750,5.09500,2.54750]),
 4: np.array([2.54750,2.54750,5.09500]),
 5: np.array([0.00000,0.00000,0.00000]),
 6: np.array([0.00000,2.54750,2.54750]),
 7: np.array([2.54750,0.00000,2.54750]),
 8: np.array([2.54750,2.54750,0.00000]),
 9: np.array([3.82125,3.82125,3.82125]),
10: np.array([6.36875,6.36875,6.36875]),
11: np.array([0.00000,1.27375,1.27375]),
12: np.array([2.54750,1.27375,1.27375]),
13: np.array([1.27375,1.27375,0.00000]),
14: np.array([1.27375,1.27375,2.54750]),
15: np.array([1.27375,0.00000,1.27375]),
16: np.array([1.27375,2.54750,1.27375]),
17: np.array([3.82125,0.00000,3.82125]),
18: np.array([3.82125,7.64250,3.82125]),
19: np.array([0.00000,3.82125,3.82125]),
20: np.array([7.64250,3.82125,3.82125]),
21: np.array([3.82125,3.82125,0.00000]),
22: np.array([3.82125,3.82125,7.64250]),
}

# GM1+ displacement used only to restore the physical 48f oxygen coordinate
d_A1g = {
11: np.array([+1,0,0]),
12: np.array([-1,0,0]),
13: np.array([0,0,+1]),
14: np.array([0,0,-1]),
15: np.array([0,+1,0]),
16: np.array([0,-1,0]),
17: np.array([0,-1,0]),
18: np.array([0,+1,0]),
19: np.array([-1,0,0]),
20: np.array([+1,0,0]),
21: np.array([0,0,-1]),
22: np.array([0,0,+1]),
}

# GM2- displacement pattern from SMODES
d_GM2m = {
 1: np.array([+1,+1,+1]),
 2: np.array([+1,-1,-1]),
 3: np.array([-1,+1,-1]),
 4: np.array([-1,-1,+1]),
 5: np.array([+1,+1,+1]),
 6: np.array([+1,-1,-1]),
 7: np.array([-1,+1,-1]),
 8: np.array([-1,-1,+1]),
11: np.array([-1,0,0]),
12: np.array([+1,0,0]),
13: np.array([0,0,-1]),
14: np.array([0,0,+1]),
15: np.array([0,-1,0]),
16: np.array([0,+1,0]),
17: np.array([0,-1,0]),
18: np.array([0,+1,0]),
19: np.array([-1,0,0]),
20: np.array([+1,0,0]),
21: np.array([0,0,-1]),
22: np.array([0,0,+1]),
}

# equilibrium positions in SMODES convention
R_eq = {}
for i in labels:
    if i in d_A1g:
        R_eq[i] = R_ref[i] + delta * d_A1g[i]
    else:
        R_eq[i] = R_ref[i].copy()

# wrap into [0,a) just for printing
def wrap_cart(r, L=10.19):
    return np.mod(r, L)

print("Equilibrium positions (SMODES indexing):")
for i in labels:
    r = wrap_cart(R_eq[i], a)
    print(f"{i:2d} {species[i]:2s}  {r[0]:8.5f} {r[1]:8.5f} {r[2]:8.5f}")

# Example frozen phonon
lam = 0.02  # Å
R_frozen = {}
for i in labels:
    d = d_GM2m.get(i, np.zeros(3))
    R_frozen[i] = R_eq[i] + lam * d

print("\nFrozen phonon positions for GM2- with lambda = 0.02 Å:")
for i in labels:
    r = wrap_cart(R_frozen[i], a)
    print(f"{i:2d} {species[i]:2s}  {r[0]:8.5f} {r[1]:8.5f} {r[2]:8.5f}")

Equilibrium positions (SMODES indexing):
 1 Y    5.09500  5.09500  5.09500
 2 Y    5.09500  2.54750  2.54750
 3 Y    2.54750  5.09500  2.54750
 4 Y    2.54750  2.54750  5.09500
 5 Ir1   0.00000  0.00000  0.00000
 6 Ir2   0.00000  2.54750  2.54750
 7 Ir3   2.54750  0.00000  2.54750
 8 Ir4   2.54750  2.54750  0.00000
 9 O    3.82125  3.82125  3.82125
10 O    6.36875  6.36875  6.36875
11 O    0.89603  1.27375  1.27375
12 O    1.65147  1.27375  1.27375
13 O    1.27375  1.27375  0.89603
14 O    1.27375  1.27375  1.65147
15 O    1.27375  0.89603  1.27375
16 O    1.27375  1.65147  1.27375
17 O    3.82125  9.29397  3.82125
18 O    3.82125  8.53853  3.82125
19 O    9.29397  3.82125  3.82125
20 O    8.53853  3.82125  3.82125
21 O    3.82125  3.82125  9.29397
22 O    3.82125  3.82125  8.53853

Frozen phonon positions for GM2- with lambda = 0.02 Å:
 1 Y    5.11500  5.11500  5.11500
 2 Y    5.11500  2.52750  2.52750
 3 Y    2.52750  5.11500  2.52750
 4 Y    2.52750  2.52750  5.11500
 5 Ir1   0.0200

In [5]:
import numpy as np

# Four Ir positions in Cartesian coordinates (angstrom)
ir_positions = {
    "Ir1": np.array([0.00000, 0.00000, 0.00000]),
    "Ir2": np.array([0.00000, 2.54750, 2.54750]),
    "Ir3": np.array([2.54750, 0.00000, 2.54750]),
    "Ir4": np.array([2.54750, 2.54750, 0.00000]),
}

def qe_angles_from_vector(v):
    """
    Convert a Cartesian vector to QE angle1, angle2.
    angle1 = polar angle from +z
    angle2 = azimuth from +x in xy plane
    Returns angles in degrees.
    """
    v = np.array(v, dtype=float)
    n = np.linalg.norm(v)
    if n == 0:
        raise ValueError("Zero vector has no direction.")
    v = v / n

    x, y, z = v
    angle1 = np.degrees(np.arccos(z))
    angle2 = np.degrees(np.arctan2(y, x))
    if angle2 < 0:
        angle2 += 360.0
    return angle1, angle2

# Tetrahedron center
center = np.mean(list(ir_positions.values()), axis=0)

print("Tetrahedron center:", center)

# All-out: moment points from center to site
print("\nAll-out AIAO directions:")
for label, r in ir_positions.items():
    v = r - center
    angle1, angle2 = qe_angles_from_vector(v)
    print(f"{label}: v = {v}, angle1 = {angle1:.12f}, angle2 = {angle2:.1f}")

# All-in: opposite directions
print("\nAll-in AIAO directions:")
for label, r in ir_positions.items():
    v = center - r
    angle1, angle2 = qe_angles_from_vector(v)
    print(f"{label}: v = {v}, angle1 = {angle1:.12f}, angle2 = {angle2:.1f}")

Tetrahedron center: [1.27375 1.27375 1.27375]

All-out AIAO directions:
Ir1: v = [-1.27375 -1.27375 -1.27375], angle1 = 125.264389682755, angle2 = 225.0
Ir2: v = [-1.27375  1.27375  1.27375], angle1 = 54.735610317245, angle2 = 135.0
Ir3: v = [ 1.27375 -1.27375  1.27375], angle1 = 54.735610317245, angle2 = 315.0
Ir4: v = [ 1.27375  1.27375 -1.27375], angle1 = 125.264389682755, angle2 = 45.0

All-in AIAO directions:
Ir1: v = [1.27375 1.27375 1.27375], angle1 = 54.735610317245, angle2 = 45.0
Ir2: v = [ 1.27375 -1.27375 -1.27375], angle1 = 125.264389682755, angle2 = 315.0
Ir3: v = [-1.27375  1.27375 -1.27375], angle1 = 125.264389682755, angle2 = 135.0
Ir4: v = [-1.27375 -1.27375  1.27375], angle1 = 54.735610317245, angle2 = 225.0


In [6]:
import re
def auto_id_positions(items):
    """
    items: [(species, [x,y,z]), ...]
    returns: [(id, species, [x,y,z])]
    
    - Keeps species exactly as given (e.g., "Mn1" stays "Mn1")
    - Adds numeric suffix only if species doesn't already end with digits
    """
    counts = {}
    out = []
    for sp, r in items:
        # Check if species already ends with digits
        if re.search(r"\d+$", sp):
            aid = sp
        else:
            counts[sp] = counts.get(sp, 0) + 1
            aid = f"{sp}{counts[sp]}"
        out.append((aid, sp, list(map(float, r))))
    return out

def make_qe_pos_block(positions, displacements=None, amp=0.0, unit="angstrom"):
    """
    positions: [(id, species, [x,y,z]), ...]
    displacements: {id: [dx,dy,dz]}  (optional)
                   or a callable  disp(id, species, pos) -> [dx,dy,dz]
    amp: scale factor for displacement
    """
    lines = [f"ATOMIC_POSITIONS {unit}"]
    use_callable = callable(displacements)
    for aid, sp, r in positions:
        r = np.array(r, float)
        if displacements is not None:
            if use_callable:
                d = np.array(displacements(aid, sp, r.copy()), float)
            else:
                d = np.array(displacements.get(aid, (0.0,0.0,0.0)), float)
            r = r + float(amp)*d
        lines.append(f" {sp:<3s} {r[0]:.12f} {r[1]:.12f} {r[2]:.12f}")
    return "\n".join(lines)

Use below for quantum ESPRESSO input files in equilibrium.

In [7]:
# Equilibrium positions from converged base/soc run (154434bc889), U=3.0 eV, AIAO, displaced O48f
positions_ang = [
    ("Y",   [5.09500000,  5.09500000,  5.09500000]),
    ("Y",   [5.09500000,  2.54750000,  2.54750000]),
    ("Y",   [2.54750000,  5.09500000,  2.54750000]),
    ("Y",   [7.64250000,  7.64250000,  5.09500000]),
    ("Ir1", [0.00000000,  0.00000000,  0.00000000]),
    ("Ir2", [5.09500000,  7.64250000,  2.54750000]),
    ("Ir3", [7.64250000,  5.09500000,  2.54750000]),
    ("Ir4", [2.54750000,  2.54750000,  0.00000000]),
    ("O",   [3.82125000,  3.82125000,  3.82125000]),   # O8b
    ("O",   [6.36875000,  6.36875000,  6.36875000]),   # O8b
    ("O",   [3.44352708,  6.36875000,  6.36875000]),   # O48f
    ("O",   [9.29397292,  6.36875000,  6.36875000]),
    ("O",   [6.36875000,  6.36875000,  3.44352708]),
    ("O",   [6.36875000,  6.36875000,  9.29397292]),
    ("O",   [6.36875000,  3.44352708,  6.36875000]),
    ("O",   [6.36875000,  9.29397292,  6.36875000]),
    ("O",   [3.82125000,  6.74647292,  3.82125000]),
    ("O",   [3.82125000,  0.89602708,  3.82125000]),
    ("O",   [6.74647292,  3.82125000,  3.82125000]),
    ("O",   [0.89602708,  3.82125000,  3.82125000]),
    ("O",   [3.82125000,  3.82125000,  6.74647292]),
    ("O",   [3.82125000,  3.82125000,  0.89602708]),
]

pos_id = auto_id_positions(positions_ang)
pos_id

[('Y1', 'Y', [5.095, 5.095, 5.095]),
 ('Y2', 'Y', [5.095, 2.5475, 2.5475]),
 ('Y3', 'Y', [2.5475, 5.095, 2.5475]),
 ('Y4', 'Y', [7.6425, 7.6425, 5.095]),
 ('Ir1', 'Ir1', [0.0, 0.0, 0.0]),
 ('Ir2', 'Ir2', [5.095, 7.6425, 2.5475]),
 ('Ir3', 'Ir3', [7.6425, 5.095, 2.5475]),
 ('Ir4', 'Ir4', [2.5475, 2.5475, 0.0]),
 ('O1', 'O', [3.82125, 3.82125, 3.82125]),
 ('O2', 'O', [6.36875, 6.36875, 6.36875]),
 ('O3', 'O', [3.44352708, 6.36875, 6.36875]),
 ('O4', 'O', [9.29397292, 6.36875, 6.36875]),
 ('O5', 'O', [6.36875, 6.36875, 3.44352708]),
 ('O6', 'O', [6.36875, 6.36875, 9.29397292]),
 ('O7', 'O', [6.36875, 3.44352708, 6.36875]),
 ('O8', 'O', [6.36875, 9.29397292, 6.36875]),
 ('O9', 'O', [3.82125, 6.74647292, 3.82125]),
 ('O10', 'O', [3.82125, 0.89602708, 3.82125]),
 ('O11', 'O', [6.74647292, 3.82125, 3.82125]),
 ('O12', 'O', [0.89602708, 3.82125, 3.82125]),
 ('O13', 'O', [3.82125, 3.82125, 6.74647292]),
 ('O14', 'O', [3.82125, 3.82125, 0.89602708])]

In [8]:
print(make_qe_pos_block(pos_id))

ATOMIC_POSITIONS angstrom
 Y   5.095000000000 5.095000000000 5.095000000000
 Y   5.095000000000 2.547500000000 2.547500000000
 Y   2.547500000000 5.095000000000 2.547500000000
 Y   7.642500000000 7.642500000000 5.095000000000
 Ir1 0.000000000000 0.000000000000 0.000000000000
 Ir2 5.095000000000 7.642500000000 2.547500000000
 Ir3 7.642500000000 5.095000000000 2.547500000000
 Ir4 2.547500000000 2.547500000000 0.000000000000
 O   3.821250000000 3.821250000000 3.821250000000
 O   6.368750000000 6.368750000000 6.368750000000
 O   3.443527080000 6.368750000000 6.368750000000
 O   9.293972920000 6.368750000000 6.368750000000
 O   6.368750000000 6.368750000000 3.443527080000
 O   6.368750000000 6.368750000000 9.293972920000
 O   6.368750000000 3.443527080000 6.368750000000
 O   6.368750000000 9.293972920000 6.368750000000
 O   3.821250000000 6.746472920000 3.821250000000
 O   3.821250000000 0.896027080000 3.821250000000
 O   6.746472920000 3.821250000000 3.821250000000
 O   0.896027080000 3.82

# GM2\u207b (A\u2082u) Phonon Displacements

SMODES irrep label: **GM2-** (A2u), degeneracy 1, 3 modes.

```text
Irrep GM2-    A2u
Degeneracy:  1
Total number of modes:   3

Mode 1: Y sub-lattice, AIAO pattern
  1 Y  [+1,+1,+1]   2 Y  [+1,-1,-1]   3 Y  [-1,+1,-1]   4 Y  [-1,-1,+1]

Mode 2: Ir sub-lattice, AIAO pattern
  5 Ir [+1,+1,+1]   6 Ir [+1,-1,-1]   7 Ir [-1,+1,-1]   8 Ir [-1,-1,+1]

Mode 3: O 48f sub-lattice
 11 O  [-1, 0, 0]  12 O  [+1, 0, 0]  13 O  [ 0, 0,-1]  14 O  [ 0, 0,+1]
 15 O  [ 0,-1, 0]  16 O  [ 0,+1, 0]  17 O  [ 0,-1, 0]  18 O  [ 0,+1, 0]
 19 O  [-1, 0, 0]  20 O  [+1, 0, 0]  21 O  [ 0, 0,-1]  22 O  [ 0, 0,+1]
```

Atoms 11\u201322 in SMODES correspond to O3\u2013O14 in `auto_id_positions` ordering (O1,O2 are the two O8b atoms). The opposite overall sign is symmetry-equivalent and corresponds to the `-Q` frozen-phonon branch.

In [9]:
amps = [0, 0.01]

# Displacement unit vectors for each GM2- mode (keyed by auto_id label)
disp_mode1 = {
    "Y1": [+1, +1, +1],
    "Y2": [+1, -1, -1],
    "Y3": [-1, +1, -1],
    "Y4": [-1, -1, +1],
}

disp_mode2 = {
    "Ir1": [+1, +1, +1],
    "Ir2": [+1, -1, -1],
    "Ir3": [-1, +1, -1],
    "Ir4": [-1, -1, +1],
}

# Mode 3: O48f sub-lattice.
# Sign determined by the Ir-Ir bridge pair, NOT by the local A1g direction.
# Rule (using the QE/154434bc889 Ir positions):
#   Ir1-Ir2 pair → +x;  Ir3-Ir4 pair → -x
#   Ir1-Ir3 pair → +y;  Ir2-Ir4 pair → -y
#   Ir1-Ir4 pair → +z;  Ir2-Ir3 pair → -z
#
# O3  bridges Ir3-Ir4 → -x   O4  bridges Ir1-Ir2 → +x
# O5  bridges Ir2-Ir3 → -z   O6  bridges Ir1-Ir4 → +z
# O7  bridges Ir2-Ir4 → -y   O8  bridges Ir1-Ir3 → +y
# O9  bridges Ir2-Ir4 → -y   O10 bridges Ir1-Ir3 → +y
# O11 bridges Ir3-Ir4 → -x   O12 bridges Ir1-Ir2 → +x
# O13 bridges Ir2-Ir3 → -z   O14 bridges Ir1-Ir4 → +z
disp_mode3 = {
    "O3":  [-1,  0,  0],
    "O4":  [+1,  0,  0],
    "O5":  [ 0,  0, -1],
    "O6":  [ 0,  0, +1],
    "O7":  [ 0, -1,  0],
    "O8":  [ 0, +1,  0],
    "O9":  [ 0, -1,  0],
    "O10": [ 0, +1,  0],
    "O11": [-1,  0,  0],
    "O12": [+1,  0,  0],
    "O13": [ 0,  0, -1],
    "O14": [ 0,  0, +1],
}

## Mode 1 — Y sub-lattice (AIAO)

In [10]:
for amp in amps:
    print(f"Amplitude = {amp} Angstrom")
    print(make_qe_pos_block(pos_id, displacements=disp_mode1, amp=amp))
    print()

Amplitude = 0 Angstrom
ATOMIC_POSITIONS angstrom
 Y   5.095000000000 5.095000000000 5.095000000000
 Y   5.095000000000 2.547500000000 2.547500000000
 Y   2.547500000000 5.095000000000 2.547500000000
 Y   7.642500000000 7.642500000000 5.095000000000
 Ir1 0.000000000000 0.000000000000 0.000000000000
 Ir2 5.095000000000 7.642500000000 2.547500000000
 Ir3 7.642500000000 5.095000000000 2.547500000000
 Ir4 2.547500000000 2.547500000000 0.000000000000
 O   3.821250000000 3.821250000000 3.821250000000
 O   6.368750000000 6.368750000000 6.368750000000
 O   3.443527080000 6.368750000000 6.368750000000
 O   9.293972920000 6.368750000000 6.368750000000
 O   6.368750000000 6.368750000000 3.443527080000
 O   6.368750000000 6.368750000000 9.293972920000
 O   6.368750000000 3.443527080000 6.368750000000
 O   6.368750000000 9.293972920000 6.368750000000
 O   3.821250000000 6.746472920000 3.821250000000
 O   3.821250000000 0.896027080000 3.821250000000
 O   6.746472920000 3.821250000000 3.821250000000
 

## Mode 2 — Ir sub-lattice (AIAO)

In [11]:
for amp in amps:
    print(f"Amplitude = {amp} Angstrom")
    print(make_qe_pos_block(pos_id, displacements=disp_mode2, amp=amp))
    print()

Amplitude = 0 Angstrom
ATOMIC_POSITIONS angstrom
 Y   5.095000000000 5.095000000000 5.095000000000
 Y   5.095000000000 2.547500000000 2.547500000000
 Y   2.547500000000 5.095000000000 2.547500000000
 Y   7.642500000000 7.642500000000 5.095000000000
 Ir1 0.000000000000 0.000000000000 0.000000000000
 Ir2 5.095000000000 7.642500000000 2.547500000000
 Ir3 7.642500000000 5.095000000000 2.547500000000
 Ir4 2.547500000000 2.547500000000 0.000000000000
 O   3.821250000000 3.821250000000 3.821250000000
 O   6.368750000000 6.368750000000 6.368750000000
 O   3.443527080000 6.368750000000 6.368750000000
 O   9.293972920000 6.368750000000 6.368750000000
 O   6.368750000000 6.368750000000 3.443527080000
 O   6.368750000000 6.368750000000 9.293972920000
 O   6.368750000000 3.443527080000 6.368750000000
 O   6.368750000000 9.293972920000 6.368750000000
 O   3.821250000000 6.746472920000 3.821250000000
 O   3.821250000000 0.896027080000 3.821250000000
 O   6.746472920000 3.821250000000 3.821250000000
 

## Mode 3 — O 48f sub-lattice

In [12]:
for amp in amps:
    print(f"Amplitude = {amp} Angstrom")
    print(make_qe_pos_block(pos_id, displacements=disp_mode3, amp=amp))
    print()

Amplitude = 0 Angstrom
ATOMIC_POSITIONS angstrom
 Y   5.095000000000 5.095000000000 5.095000000000
 Y   5.095000000000 2.547500000000 2.547500000000
 Y   2.547500000000 5.095000000000 2.547500000000
 Y   7.642500000000 7.642500000000 5.095000000000
 Ir1 0.000000000000 0.000000000000 0.000000000000
 Ir2 5.095000000000 7.642500000000 2.547500000000
 Ir3 7.642500000000 5.095000000000 2.547500000000
 Ir4 2.547500000000 2.547500000000 0.000000000000
 O   3.821250000000 3.821250000000 3.821250000000
 O   6.368750000000 6.368750000000 6.368750000000
 O   3.443527080000 6.368750000000 6.368750000000
 O   9.293972920000 6.368750000000 6.368750000000
 O   6.368750000000 6.368750000000 3.443527080000
 O   6.368750000000 6.368750000000 9.293972920000
 O   6.368750000000 3.443527080000 6.368750000000
 O   6.368750000000 9.293972920000 6.368750000000
 O   3.821250000000 6.746472920000 3.821250000000
 O   3.821250000000 0.896027080000 3.821250000000
 O   6.746472920000 3.821250000000 3.821250000000
 